In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
import scipy
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
train = ImageDataGenerator(rescale=1./255)

data = train.flow_from_directory(
    "train",
    target_size=(24,24),
    batch_size=32,
    class_mode='binary'
)

Found 1357 images belonging to 2 classes.


In [3]:
model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(24,24,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [4]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [5]:
history = model.fit(data, epochs=10)

Epoch 1/10
43/43 [==============================] - 33s 726ms/step - loss: 0.4039 - accuracy: 0.8254
Epoch 2/10
43/43 [==============================] - 7s 153ms/step - loss: 0.1667 - accuracy: 0.9425
Epoch 3/10
43/43 [==============================] - 7s 155ms/step - loss: 0.1203 - accuracy: 0.9573
Epoch 4/10
43/43 [==============================] - 7s 152ms/step - loss: 0.0890 - accuracy: 0.9654
Epoch 5/10
43/43 [==============================] - 7s 151ms/step - loss: 0.0667 - accuracy: 0.9742
Epoch 6/10
43/43 [==============================] - 7s 154ms/step - loss: 0.0531 - accuracy: 0.9831
Epoch 7/10
43/43 [==============================] - 6s 150ms/step - loss: 0.0521 - accuracy: 0.9831
Epoch 8/10
43/43 [==============================] - 7s 150ms/step - loss: 0.0499 - accuracy: 0.9838
Epoch 9/10
43/43 [==============================] - 6s 150ms/step - loss: 0.0360 - accuracy: 0.9875
Epoch 10/10
43/43 [==============================] - 6s 149ms/step - loss: 0.0356 - accuracy: 0.986

In [9]:
model.save("drowsiness_model.h5")

In [17]:
import cv2
import numpy as np
from playsound import playsound

In [15]:
model = load_model("drowsiness_model.h5")

In [18]:
faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

In [31]:
cap = cv2.VideoCapture(0)

closed_count = 0
alarm_triggered = False

# ---------------- LOOP ----------------
while True:

    ret, frame = cap.read()
    if not ret:
        break

    # reduce lag (fix freezing)
    frame = cv2.resize(frame, (640, 480))

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    for (x, y, w, h) in faces:

        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        # eye regions
        left_eye = frame[
            y + int(h * 0.20): y + int(h * 0.45),
            x + int(w * 0.10): x + int(w * 0.45)
        ]

        right_eye = frame[
            y + int(h * 0.20): y + int(h * 0.45),
            x + int(w * 0.55): x + int(w * 0.90)
        ]

        # default predictions
        left_pred = 1
        right_pred = 1

        # LEFT EYE PREDICTION
        try:
            left_img = cv2.resize(left_eye, (24, 24))
            left_img = left_img / 255.0
            left_img = np.reshape(left_img, (1, 24, 24, 3))
            left_pred = model.predict(left_img, verbose=0)[0][0]
        except:
            pass

        # RIGHT EYE PREDICTION
        try:
            right_img = cv2.resize(right_eye, (24, 24))
            right_img = right_img / 255.0
            right_img = np.reshape(right_img, (1, 24, 24, 3))
            right_pred = model.predict(right_img, verbose=0)[0][0]
        except:
            pass

        # ---------------- FIXED LOGIC ----------------
        avg_pred = (left_pred + right_pred) / 2

        if avg_pred < 0.5:
            closed_count += 1
            label = "Closed Eyes"
            color = (0, 0, 255)
        else:
            closed_count = 0
            alarm_triggered = False
            label = "Open Eyes"
            color = (0, 255, 0)

        # ---------------- ALERT ----------------
        if closed_count > 20 and not alarm_triggered:

            cv2.putText(
                frame,
                "DROWSINESS ALERT!",
                (100, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                3
            )

            alarm_triggered = True
            playsound("Alarm.mp3")

        # show label
        cv2.putText(
            frame,
            label,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

    cv2.imshow("Driver Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [9]:
cap.release()
cv2.destroyAllWindows()